# Análise de frota
Demonstração de portfólio com dados inteiramente fictícios. O objetivo é consolidar o consumo por veículo e priorizar inspeções de pneus em uma data de referência fixa. Execute as células em ordem usando Python 3.10 ou superior.

In [1]:
from collections import defaultdict
from datetime import date
from decimal import Decimal

# Cada registro representa quilômetros percorridos e litros consumidos em um período.
consumo_por_periodo = [
    ("F-101", 440, "160"),
    ("F-101", 310, "110"),
    ("F-102", 360, "150"),
    ("F-102", 280, "120"),
    ("F-103", 500, "170"),
    ("F-103", 410, "140"),
    ("F-104", 390, "145"),
    ("F-104", 325, "125"),
]

ultima_inspecao = {
    "F-101": date(2026, 11, 11),
    "F-102": date(2026, 10, 17),
    "F-103": date(2026, 10, 27),
    "F-104": None,
}

data_referencia = date(2026, 11, 26)
intervalo_inspecao_dias = 30


## Consumo por veículo
A eficiência é a soma dos quilômetros dividida pela soma dos litros. A linha da frota usa os mesmos totais, ponderados pelo consumo de cada veículo. Neste exemplo, todos os veículos tiveram consumo positivo.

In [2]:
km_por_veiculo = defaultdict(int)
litros_por_veiculo = defaultdict(Decimal)

for veiculo, km, litros in consumo_por_periodo:
    km_por_veiculo[veiculo] += km
    litros_por_veiculo[veiculo] += Decimal(litros)

eficiencia_por_veiculo = {
    veiculo: km / litros_por_veiculo[veiculo]
    for veiculo, km in km_por_veiculo.items()
}

print(f"{'Veículo':<10} {'Distância (km)':>15} {'Consumo (L)':>14} {'km/L':>8}")
for veiculo in sorted(eficiencia_por_veiculo):
    distancia = km_por_veiculo[veiculo]
    consumo = litros_por_veiculo[veiculo]
    eficiencia = eficiencia_por_veiculo[veiculo]
    print(f"{veiculo:<10} {distancia:>15} {consumo:>14.1f} {eficiencia:>8.2f}")

km_total = sum(km_por_veiculo.values())
litros_total = sum(litros_por_veiculo.values())
eficiencia_total = km_total / litros_total
print(f"{'Frota':<10} {km_total:>15} {litros_total:>14.1f} {eficiencia_total:>8.2f}")


Veículo     Distância (km)    Consumo (L)     km/L
F-101                  750          270.0     2.78
F-102                  640          270.0     2.37
F-103                  910          310.0     2.94
F-104                  715          270.0     2.65
Frota                 3015         1120.0     2.69


## Programação de inspeções de pneus
O prazo de 30 dias é uma premissa da demonstração. A lista prioriza veículos sem registro de inspeção e, em seguida, os veículos com o maior número de dias desde a última inspeção. Uma inspeção vence ao completar 30 dias.

In [3]:
dias_por_veiculo = {
    veiculo: (data_referencia - data_inspecao).days if data_inspecao else None
    for veiculo, data_inspecao in ultima_inspecao.items()
}

def ordem_de_inspecao(item):
    veiculo, dias = item
    if dias is None:
        return (0, 0, veiculo)
    return (1, -dias, veiculo)

for veiculo, dias in sorted(dias_por_veiculo.items(), key=ordem_de_inspecao):
    if dias is None:
        print(f"{veiculo}: sem registro")
        continue
    situacao = "vencida" if dias >= intervalo_inspecao_dias else "em dia"
    print(f"{veiculo}: {dias} dias — {situacao}")


F-104: sem registro
F-102: 40 dias — vencida
F-103: 30 dias — vencida
F-101: 15 dias — em dia


## Leitura dos resultados
- A frota fictícia percorreu **3.015 km** e consumiu **1.120 L**, com eficiência de **2,69 km/L**.
- O veículo **F-102** teve a menor eficiência da amostra, **2,37 km/L**. Um acompanhamento real precisaria comparar veículos e operações equivalentes, considerando carga e trajeto.
- A programação começa com **F-104**, que está sem registro. Em seguida vêm **F-102** e **F-103**, ambos com a inspeção vencida na data de referência.